# distfuzz on two GPUs (Kaggle 2x T4)

Runs three distfuzz campaigns on NCCL with one GPU per rank, then packs the results into `/kaggle/working/gpufuzz.tgz`.

Notebook settings: Accelerator **GPU T4 x2**, Internet **on**. Use **Save Version > Save & Run All (Commit)** so the outputs under `/kaggle/working` are kept after the session ends.

| arm | what it runs | default minutes |
|---|---|---|
| `collectives_nccl` | collectives fuzzer, random mode, 25 % of programs rank-divergent, NCCL with CUDA tensors | 45 |
| `dtensor_cuda` | DTensor fuzzer, guided mode, meshes on CUDA over NCCL, values checked against a CPU reference | 45 |
| `train_cuda` | training profile, workers on NCCL, CPU reference, known-bug predictor at 5 % | 45 |

Change the budget in cell 3. Set an arm's minutes to 0 to skip it. With the defaults the notebook needs about 2.5 hours of GPU time.

See `docs/examples/kaggle-two-gpus.md` in the repository for how to read the results.

In [ ]:
# (1) GPU, driver, torch 2.14.0. Every fuzzer runs in a subprocess, so no kernel restart is needed after the install.
import subprocess
import sys

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
drv = subprocess.check_output(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]).decode().split()[0]
CU = "cu130" if int(drv.split(".")[0]) >= 580 else "cu126"  # cu130 needs driver R580+; both wheels ship sm_75
print("python", sys.version.split()[0], "| driver", drv, "-> torch 2.14.0 +", CU)
if sys.version_info < (3, 12):
    print("Python < 3.12 has no sys.monitoring: the fuzzers run without coverage (guided mode behaves like random)")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch==2.14.0",
        "--index-url",
        f"https://download.pytorch.org/whl/{CU}",
    ],
    check=True,
)
probe = (
    "import torch; print(torch.__version__, 'cuda', torch.version.cuda, torch.cuda.get_arch_list(), "
    "'gpus', torch.cuda.device_count(), torch.cuda.get_device_name(0), 'nccl', torch.cuda.nccl.version())"
)
TORCH_INFO = subprocess.check_output([sys.executable, "-c", probe]).decode().strip()
print(TORCH_INFO)
assert "gpus 2" in TORCH_INFO, "need two visible GPUs: set Accelerator to GPU T4 x2"

In [ ]:
# (2) distfuzz from GitHub. Pin REF to a commit or tag to repeat a run exactly.
import os
import shutil

REF = "main"
REPO = "/kaggle/working/distfuzz"
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "-q", "https://github.com/Temilola23/distfuzz.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", REF], check=True)
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "--short", "HEAD"]).decode().strip()
print("distfuzz", REF, "@", COMMIT)

In [ ]:
# (3) Output directory, budget, seeds and the runner. Change minutes here, nothing else.
import json
import time

OUT = "/kaggle/working/gpufuzz"
os.makedirs(OUT, exist_ok=True)
MINUTES = {"collectives_nccl": 45, "dtensor_cuda": 45, "train_cuda": 45}
SEEDS = {"collectives_nccl": 111, "dtensor_cuda": 211, "train_cuda": 501}
# The fuzzers import distfuzz from the clone through PYTHONPATH. Do not export TORCH_NCCL_USE_COMM_NONBLOCKING here:
# the collectives executor turns non-blocking communicators on for its own groups with a matching timeout, and in
# the DTensor and training workers non-blocking mode makes every equal-split all_to_all_single fail (bug §76).
ENV = dict(os.environ, PYTHONPATH=f"{REPO}/src", PYTHONUNBUFFERED="1")
NCCL_TIMEOUT = []  # e.g. ["--timeout", "10"]; the smoke cell raises it if healthy collectives time out
with open(f"{OUT}/env.txt", "w") as fh:
    fh.write(f"{TORCH_INFO}\ndriver {drv}\ndistfuzz {REF}@{COMMIT}\nseeds {SEEDS} minutes {MINUTES}\n")


def cleanup():
    # rank processes are daemonic children; if `timeout` had to kill a fuzzer they would outlive it and hold the GPUs
    for pid in os.listdir("/proc"):
        try:
            cmd = open(f"/proc/{pid}/cmdline", "rb").read().decode(errors="replace")
            if pid.isdigit() and int(pid) != os.getpid() and ("multiprocessing.spawn" in cmd or "distfuzz." in cmd):
                os.kill(int(pid), 9)
        except (OSError, ValueError):
            pass
    time.sleep(2)


def run(name, args, minutes, env=None):
    """Run one fuzzer as `python -m <args>` with a hard wall-clock cap and print the tail of its log."""
    log = f"{OUT}/{name}.log"
    t = time.time()
    cap = int(minutes * 60 + 600)  # the fuzzer stops itself at `minutes`; this only catches a wedged parent
    with open(log, "w") as fh:
        p = subprocess.run(
            ["timeout", "--kill-after=60", str(cap), sys.executable, "-m", *args],
            stdout=fh,
            stderr=subprocess.STDOUT,
            env=env or ENV,
        )
    cleanup()
    state = "KILLED by cap" if p.returncode in (124, 137) else "ok" if p.returncode == 0 else "error"
    print(f"== {name}: exit {p.returncode} after {(time.time() - t) / 60:.1f} min ({state})")
    print("".join(open(log).readlines()[-6:]))
    return p.returncode

In [ ]:
# (4) Smoke test, 30 s each: NCCL collectives with no rank-divergent programs, then DTensor on CUDA.
# If NCCL fails on this machine, retry once with peer-to-peer off (the usual fix on dual-T4 machines).
def smoke_collectives(tag):
    out = f"{OUT}/smoke_collectives_{tag}"
    args = [
        "distfuzz.collectives",
        "fuzz",
        "--mode",
        "random",
        "--backend",
        "nccl",
        "--world-size",
        "2",
        "--gen-diverge",
        "0",
        "--minutes",
        "0.5",
        "--seed",
        "1",
        "--out",
        out,
    ]
    rc = run(f"smoke_collectives_{tag}", args, 0.5)
    try:
        return rc, json.load(open(f"{out}/summary.json"))["final"]
    except (OSError, ValueError, KeyError):
        return rc, None


rc, s = smoke_collectives("p2p")
if rc != 0 or not s or s["ok"] == 0:
    ENV["NCCL_P2P_DISABLE"] = "1"
    print("smoke test failed with P2P on; retrying with NCCL_P2P_DISABLE=1")
    rc, s = smoke_collectives("nop2p")
assert rc == 0 and s and s["ok"] > 0, f"NCCL smoke test failed (rc={rc}): read {OUT}/smoke_collectives_*.log"
print(json.dumps(s))
if s["unconfirmed"]:  # a timed-out collective that finished at the longer confirm timeout was slow, not stuck
    NCCL_TIMEOUT = ["--timeout", "10"]
    print("healthy collectives timed out at the default: using", NCCL_TIMEOUT)

rc = run(
    "smoke_dtensor",
    [
        "distfuzz.dtensor",
        "fuzz",
        "--device",
        "cuda",
        "--world-size",
        "2",
        "--mode",
        "random",
        "--minutes",
        "0.5",
        "--seed",
        "1",
        "--out",
        f"{OUT}/smoke_dtensor",
    ],
    0.5,
)
p = f"{OUT}/smoke_dtensor/stats.jsonl"
st = [json.loads(x) for x in open(p)] if os.path.exists(p) else []
print(st[-1] if st else "no DTensor stats")
assert rc == 0 and st and st[-1]["execs"] > 0, f"DTensor smoke test failed: read {OUT}/smoke_dtensor.log"
with open(f"{OUT}/env.txt", "a") as fh:
    fh.write(f"NCCL_P2P_DISABLE={ENV.get('NCCL_P2P_DISABLE', '')} timeout_args={NCCL_TIMEOUT}\n")

In [ ]:
# (5) Arm 1: collectives fuzzer on NCCL, random mode, 25 % of generated programs rank-divergent.
M, S = MINUTES["collectives_nccl"], SEEDS["collectives_nccl"]
if M:
    args = "distfuzz.collectives fuzz --mode random --backend nccl --world-size 2 --gen-diverge 0.25"
    args += f" --minutes {M} --seed {S}"
    run("collectives_nccl", args.split() + ["--out", f"{OUT}/collectives_nccl", *NCCL_TIMEOUT], M)

In [ ]:
# (6) Arm 2: DTensor fuzzer with the meshes on CUDA over NCCL; every value is checked against a CPU reference.
M, S = MINUTES["dtensor_cuda"], SEEDS["dtensor_cuda"]
if M:
    args = f"distfuzz.dtensor fuzz --device cuda --world-size 2 --mode guided --minutes {M} --seed {S}"
    run("dtensor_cuda", args.split() + ["--out", f"{OUT}/dtensor_cuda"], M)

In [ ]:
# (7) Arm 3: training profile on NCCL, two ranks, CPU reference. DCP_PG_TIMEOUT above --timeout lets the harness
# report a stuck collective with stack locations. Under cuda the generator skips async thread saves, and float8
# runs only on GPUs with compute capability 8.9 or newer.
M, S = MINUTES["train_cuda"], SEEDS["train_cuda"]
if M:
    args = "distfuzz.dcp fuzz --profile train --device cuda --max-w 2 --timeout 60 --known-rate 0.05"
    args += f" --minutes {M} --seed {S}"
    run("train_cuda", args.split() + ["--out", f"{OUT}/train_cuda"], M, env=dict(ENV, DCP_PG_TIMEOUT="120"))

In [ ]:
# (8) Summary of every arm, then one tarball with findings, stats.jsonl, logs and SUMMARY.json.
summary = {"env": open(f"{OUT}/env.txt").read()}
p = f"{OUT}/collectives_nccl/summary.json"
if os.path.exists(p):
    s = json.load(open(p))
    f = s["final"]
    summary["collectives_nccl"] = s
    print(
        f"## collectives_nccl: execs={f['execs']} t={f['t']}s ok={f['ok']} crash={f['crash']} hang={f['hang']} "
        f"checked={f['checked']} restarts={f.get('restarts')}"
    )
    for r in sorted(s["findings"], key=lambda r: -r["count"]):
        print(f"  {r['count']:5d}  {r['sig'][:150]}")
for arm in ("dtensor_cuda", "train_cuda"):
    p = f"{OUT}/{arm}/findings.jsonl"
    if not os.path.exists(p):
        continue
    recs = [json.loads(x) for x in open(p)]
    st = [json.loads(x) for x in open(f"{OUT}/{arm}/stats.jsonl")]
    summary[arm] = {"final": st[-1] if st else None, "sigs": [r["sig"] for r in recs]}
    print(f"\n## {arm}: {st[-1] if st else 'no stats'}")
    for r in recs:
        if r["finding"]["kind"] not in ("FUZZER_INPUT_ERR", "DIST_ERR", "CUDA_UNSUPPORTED"):
            print("  ", r["sig"][:150])
json.dump(summary, open(f"{OUT}/SUMMARY.json", "w"), indent=1, default=str)
subprocess.run(["tar", "czf", "/kaggle/working/gpufuzz.tgz", "-C", "/kaggle/working", "gpufuzz"], check=True)
print("\n", subprocess.check_output(["du", "-sh", "/kaggle/working/gpufuzz.tgz"]).decode())